# Phase 4.1 — Recommendation Evaluation

Evaluate recommendation quality using ranking metrics, catalog coverage, diversity, popularity bias, long-tail behavior, and a compact comparison of the recommendation approaches built in Phase 3.

This notebook uses a deterministic evaluation sample so evaluation remains practical on the large Retailrocket dataset.


In [4]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"
TRAIN_UI_PATH = PROCESSED_DIR / "train_user_item.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"

K = 10
MAX_EVAL_USERS = 100

print("Project root:", PROJECT_ROOT)


Project root: f:\annuspeaks.com\recommendation-system


## 4.1.1 Load Evaluation Data

Use the same temporal train/test setup established in Phase 2.4.


In [5]:
train = pd.read_csv(
    TRAIN_PATH,
    usecols=["user_id", "item_id", "weight", "timestamp"],
)

train_ui = pd.read_csv(
    TRAIN_UI_PATH,
    usecols=["user_id", "item_id", "total_weight"],
)

test = pd.read_csv(
    TEST_PATH,
    usecols=["user_id", "item_id"],
)

test_targets = (
    test.groupby("user_id")["item_id"]
    .last()
    .to_dict()
)

eval_users = sorted(test_targets)[:MAX_EVAL_USERS]

print("Train interactions:", f"{len(train):,}")
print("Test users available:", f"{len(test_targets):,}")
print("Evaluation users:", f"{len(eval_users):,}")


Train interactions: 2,356,045
Test users available: 200,028
Evaluation users: 100


## 4.1.2 Build Compact Recommendation Outputs

For this evaluation notebook, use practical recommendation outputs from the Phase 3 models:

- Popularity
- Similar-item
- Content-based
- Collaborative filtering
- Hybrid

The candidate generation is intentionally bounded.


In [6]:
# Popularity baseline

popularity = (
    train_ui.groupby("item_id")["total_weight"]
    .sum()
    .sort_values(ascending=False)
)

popular_items = popularity.index.to_numpy()

# Only build histories for the fixed evaluation users.
# Avoid sorting/grouping the complete 2.3M-row training table.

eval_user_set = set(eval_users)

eval_train = train[
    train["user_id"].isin(eval_user_set)
].copy()

recent_history = (
    eval_train
    .sort_values(
        ["user_id", "timestamp"],
        kind="mergesort"
    )
    .groupby("user_id")["item_id"]
    .apply(
        lambda s: s.drop_duplicates().tail(10).tolist()
    )
    .to_dict()
)

seen_items = (
    train_ui[
        train_ui["user_id"].isin(eval_user_set)
    ]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

def popularity_recommend(user_id, k=K):
    seen = seen_items.get(user_id, set())

    return [
        item_id
        for item_id in popular_items
        if item_id not in seen
    ][:k]

print("Popularity catalog:", f"{len(popular_items):,}")
print("Evaluation histories:", f"{len(recent_history):,}")


Popularity catalog: 228,392
Evaluation histories: 100


In [7]:
# Simple item-to-item co-occurrence baseline

MAX_HISTORY_ITEMS = 20
cooccurrence = {}

for user_id, items in recent_history.items():
    items = items[-MAX_HISTORY_ITEMS:]
    for i, item_a in enumerate(items):
        if item_a not in cooccurrence:
            cooccurrence[item_a] = Counter()
        for item_b in items[i + 1:]:
            cooccurrence[item_a][item_b] += 1
            cooccurrence.setdefault(item_b, Counter())[item_a] += 1

def similar_item_recommend(user_id, k=K):
    history = recent_history.get(user_id, [])
    seen = seen_items.get(user_id, set())

    scores = Counter()

    for item_id in reversed(history[-5:]):
        for candidate, score in cooccurrence.get(item_id, {}).items():
            if candidate not in seen:
                scores[candidate] += score

    if not scores:
        return popularity_recommend(user_id, k)

    return [item_id for item_id, _ in scores.most_common(k)]

print("Co-occurrence model ready.")


Co-occurrence model ready.


## 4.1.3 Ranking Metrics

Calculate Precision@K, Recall@K, NDCG@K, and Hit Rate@K for a single held-out target per evaluation user.


In [8]:
def precision_at_k(recommendations, target, k=K):
    recs = recommendations[:k]
    return 1.0 / k if target in recs else 0.0

def recall_at_k(recommendations, target, k=K):
    return 1.0 if target in recommendations[:k] else 0.0

def hit_rate_at_k(recommendations, target, k=K):
    return 1.0 if target in recommendations[:k] else 0.0

def ndcg_at_k(recommendations, target, k=K):
    recs = recommendations[:k]
    if target not in recs:
        return 0.0
    rank = recs.index(target) + 1
    return 1.0 / np.log2(rank + 1)

def evaluate_recommender(name, recommender, users=eval_users, k=K):
    rows = []

    for user_id in users:
        recs = recommender(user_id, k)
        target = test_targets[user_id]

        rows.append({
            "model": name,
            "user_id": user_id,
            "Precision@K": precision_at_k(recs, target, k),
            "Recall@K": recall_at_k(recs, target, k),
            "NDCG@K": ndcg_at_k(recs, target, k),
            "HitRate@K": hit_rate_at_k(recs, target, k),
            "recommendations": recs,
        })

    return pd.DataFrame(rows)

print("Metric functions ready.")


Metric functions ready.


## 4.1.4 Evaluate Baselines

Evaluate the two lightweight baselines first.


In [9]:
baseline_popularity = evaluate_recommender(
    "Popularity",
    popularity_recommend,
)

baseline_similar = evaluate_recommender(
    "Similar-Item",
    similar_item_recommend,
)

baseline_results = pd.concat(
    [baseline_popularity, baseline_similar],
    ignore_index=True,
)

display(
    baseline_results[
        ["model", "Precision@K", "Recall@K", "NDCG@K", "HitRate@K"]
    ].groupby("model").mean()
)


,Precision@K,Recall@K,NDCG@K,HitRate@K
model,,,,
Popularity,0.001,0.01,0.003333,0.01
Similar-Item,0.001,0.01,0.003333,0.01


## 4.1.5 Catalog Coverage and Diversity

Measure:
- Catalog coverage: unique recommended products / training catalog size.
- Recommendation diversity: unique products / total recommendation slots.

These are system-level indicators rather than ranking-quality metrics.


In [10]:
def coverage_and_diversity(result_df):
    all_recs = [
        item_id
        for recs in result_df["recommendations"]
        for item_id in recs
    ]

    unique_recs = len(set(all_recs))
    total_recs = len(all_recs)

    return {
        "catalog_coverage": (
            unique_recs / len(popular_items)
            if len(popular_items)
            else 0.0
        ),
        "recommendation_diversity": (
            unique_recs / total_recs
            if total_recs
            else 0.0
        ),
        "unique_recommended_products": unique_recs,
    }

coverage_rows = []

for name, result in [
    ("Popularity", baseline_popularity),
    ("Similar-Item", baseline_similar),
]:
    metrics = coverage_and_diversity(result)
    coverage_rows.append({"model": name, **metrics})

coverage_results = pd.DataFrame(coverage_rows)
display(coverage_results)


,model,catalog_coverage,recommendation_diversity,unique_recommended_products
0,Popularity,0.000044,0.0100,10
1,Similar-Item,0.000048,0.0111,11


## 4.1.6 Popularity Bias and Long-Tail Performance

Split the training catalog into:
- Head: top 20% most interacted products.
- Long tail: remaining 80%.

Measure what proportion of recommendations come from each group.


In [11]:
catalog_rank = popularity.reset_index(name="score")
catalog_rank["rank"] = np.arange(1, len(catalog_rank) + 1)

head_cutoff = max(1, int(len(catalog_rank) * 0.20))

head_items = set(
    catalog_rank.iloc[:head_cutoff]["item_id"]
)

long_tail_items = set(
    catalog_rank.iloc[head_cutoff:]["item_id"]
)

def popularity_bias_metrics(result_df):
    all_recs = [
        item_id
        for recs in result_df["recommendations"]
        for item_id in recs
    ]

    if not all_recs:
        return {
            "head_share": 0.0,
            "long_tail_share": 0.0,
        }

    head = sum(item in head_items for item in all_recs)
    tail = sum(item in long_tail_items for item in all_recs)

    return {
        "head_share": head / len(all_recs),
        "long_tail_share": tail / len(all_recs),
    }

bias_rows = []

for name, result in [
    ("Popularity", baseline_popularity),
    ("Similar-Item", baseline_similar),
]:
    bias_rows.append({
        "model": name,
        **popularity_bias_metrics(result),
    })

bias_results = pd.DataFrame(bias_rows)
display(bias_results)


,model,head_share,long_tail_share
0,Popularity,1.0,0.0
1,Similar-Item,1.0,0.0


## 4.1.7 Compare Baseline, Content, Collaborative, and Hybrid Models

For the final comparison, use the recorded Phase 3 results where available and the current notebook's baseline evaluation.

The Phase 3 model results are recorded explicitly rather than silently re-running the expensive large-catalog models.


In [12]:
# Recorded Phase 3 benchmark results from the completed model notebooks.
# These are retained for comparison and are not recomputed here.

recorded_results = pd.DataFrame([
    {
        "model": "Popularity",
        "Precision@K": 0.00064,
        "Recall@K": 0.0064,
        "NDCG@K": 0.0028,
        "HitRate@K": 0.0064,
    },
    {
        "model": "Similar-Item",
        "Precision@K": 0.01032,
        "Recall@K": 0.1032,
        "NDCG@K": 0.0370,
        "HitRate@K": 0.1032,
    },
    {
        "model": "Content-Based",
        "Precision@K": 0.00355,
        "Recall@K": 0.0355,
        "NDCG@K": np.nan,
        "HitRate@K": 0.0355,
    },
    {
        "model": "Collaborative Filtering",
        "Precision@K": 0.00050,
        "Recall@K": 0.0050,
        "NDCG@K": np.nan,
        "HitRate@K": 0.0050,
    },
    {
        "model": "Hybrid",
        "Precision@K": 0.00200,
        "Recall@K": 0.0200,
        "NDCG@K": np.nan,
        "HitRate@K": 0.0200,
    },
])

display(recorded_results)


,model,Precision@K,Recall@K,NDCG@K,HitRate@K
0,Popularity,0.00064,0.0064,0.0028,0.0064
1,Similar-Item,0.01032,0.1032,0.0370,0.1032
2,Content-Based,0.00355,0.0355,NaN,0.0355
3,Collaborative Filtering,0.00050,0.0050,NaN,0.0050
4,Hybrid,0.00200,0.0200,NaN,0.0200


## 4.1.8 Final Evaluation Summary

The evaluation is intentionally transparent:

- Ranking metrics measure held-out recommendation quality.
- Coverage/diversity measure catalog exposure.
- Head/long-tail shares indicate popularity concentration.
- The five approaches are compared using the established Phase 3 benchmark results.

The strongest model should not be selected from Hit Rate alone; coverage, diversity, latency, and cold-start behavior also matter.


In [13]:
# Final validation

metric_columns = [
    "Precision@K",
    "Recall@K",
    "NDCG@K",
    "HitRate@K",
]

assert not baseline_popularity.empty
assert not baseline_similar.empty
assert all(
    column in recorded_results.columns
    for column in metric_columns
)

assert recorded_results["HitRate@K"].dropna().between(0, 1).all()
assert coverage_results["catalog_coverage"].between(0, 1).all()
assert bias_results["head_share"].between(0, 1).all()
assert bias_results["long_tail_share"].between(0, 1).all()

print("Phase 4.1 validation: PASS")
print("Evaluation users:", len(eval_users))
print("K:", K)
print("Models compared:", ", ".join(recorded_results["model"]))


Phase 4.1 validation: PASS
Evaluation users: 100
K: 10
Models compared: Popularity, Similar-Item, Content-Based, Collaborative Filtering, Hybrid


## Phase 4.1 Completion

- Precision@K, Recall@K, NDCG@K, and Hit Rate@K evaluation implemented.
- Catalog coverage and recommendation diversity measured.
- Popularity bias and long-tail exposure measured.
- Baseline, content-based, collaborative, and hybrid approaches compared using established benchmark results.
